In [1]:
!pip install -U transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 91.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 27.6 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1

In [2]:
# ============================================================
# CONCAT MODEL — 5-SEED RETRAINING (Section 4.6)
# Run this notebook on Kaggle with GPU T4 x2 (separate notebook from SCAF)
# Estimated time: ~5-6 hours for all 5 seeds
# ============================================================

# ------------------------------------------------------------
# 0) Imports
# ------------------------------------------------------------
import time
import json
import os
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ------------------------------------------------------------
# 1) CONFIG — UPDATE THIS PATH TO YOUR DATASET
# ------------------------------------------------------------
DATA_PATH  = "/kaggle/input/datasets/kazitasnianitu/imdb-dataset/IMDB_Cleaned (1).csv"
MODEL_NAME = "google/electra-base-discriminator"
TFIDF_DIM  = 3000
SEEDS      = [42, 123, 456, 789, 1011]
RESULTS_FILE = "/kaggle/working/concat_5seed_results.json"

# ------------------------------------------------------------
# 2) Load Data (same split every time)
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
text_column = "cleaned_review" if "cleaned_review" in df.columns else "review"
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts  = df[text_column].astype(str).tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

train_texts  = texts[:40000];  train_labels  = labels[:40000]
val_texts    = texts[40000:45000]; val_labels = labels[40000:45000]
test_texts   = texts[45000:50000]; test_labels = labels[45000:50000]

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

# ------------------------------------------------------------
# 3) TF-IDF
# ------------------------------------------------------------
tfidf = TfidfVectorizer(max_features=TFIDF_DIM)
tfidf.fit(train_texts)

train_tfidf = tfidf.transform(train_texts).toarray()
val_tfidf   = tfidf.transform(val_texts).toarray()
test_tfidf  = tfidf.transform(test_texts).toarray()

# ------------------------------------------------------------
# 4) Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts_list):
    return tokenizer(texts_list, truncation=True, padding=True, max_length=512)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ------------------------------------------------------------
# 5) Dataset Class
# ------------------------------------------------------------
class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, tfidf_features, labels):
        self.encodings = encodings
        self.tfidf     = tfidf_features
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["tfidf"]  = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HybridDataset(train_enc, train_tfidf, train_labels)
val_dataset   = HybridDataset(val_enc,   val_tfidf,   val_labels)
test_dataset  = HybridDataset(test_enc,  test_tfidf,  test_labels)

# ------------------------------------------------------------
# 6) Concat Model Definition (matches your saved architecture)
# ------------------------------------------------------------
class ElectraTFIDF(nn.Module):
    def __init__(self, model_name, tfidf_dim, num_labels=2):
        super().__init__()
        self.electra  = AutoModel.from_pretrained(model_name)
        hidden_size   = self.electra.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + tfidf_dim, 512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_labels)
        )

    def forward(self, input_ids, attention_mask, tfidf, labels=None, token_type_ids=None):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        cls_output = outputs.last_hidden_state[:, 0, :].contiguous()
        combined   = torch.cat((cls_output, tfidf), dim=1)
        logits     = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

# ------------------------------------------------------------
# 7) Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# ------------------------------------------------------------
# 8) MAIN LOOP — Train for each seed
# ------------------------------------------------------------
all_results = []

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        all_results = json.load(f)
    completed_seeds = [r["seed"] for r in all_results]
    print(f"Resuming. Already completed seeds: {completed_seeds}")
else:
    completed_seeds = []

for seed in SEEDS:
    if seed in completed_seeds:
        print(f"\nSeed {seed} already done, skipping...")
        continue

    print(f"\n{'='*60}")
    print(f"TRAINING SEED {seed}")
    print(f"{'='*60}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = ElectraTFIDF(MODEL_NAME, TFIDF_DIM, num_labels=2)

    training_args = TrainingArguments(
        output_dir                  = f"./results_seed{seed}",
        learning_rate               = 2e-5,
        lr_scheduler_type           = "cosine",
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 16,
        num_train_epochs            = 4,
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        warmup_steps                = 500,
        eval_strategy               = "epoch",
        save_strategy               = "no",
        load_best_model_at_end      = False,
        fp16                        = True,
        logging_steps               = 200,
        report_to                   = "none",
        seed                        = seed,
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
    )

    start_time = time.time()
    trainer.train()
    train_time = (time.time() - start_time) / 60

    test_results = trainer.predict(test_dataset)
    test_preds   = np.argmax(test_results.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='binary'
    )
    acc = accuracy_score(test_labels, test_preds)

    result = {
        "seed": seed,
        "test_accuracy": float(acc),
        "test_precision": float(precision),
        "test_recall": float(recall),
        "test_f1": float(f1),
        "train_time_min": train_time,
    }
    all_results.append(result)

    print(f"\nSeed {seed} DONE: Acc={acc:.4f}, Time={train_time:.1f}min")

    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)

    del model, trainer
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 9) FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("CONCAT — 5-SEED FINAL RESULTS")
print("=" * 60)

accs = [r["test_accuracy"] for r in all_results]
for r in all_results:
    print(f"Seed {r['seed']:5d} -> Acc: {r['test_accuracy']:.4f}")

mean_acc = np.mean(accs)
std_acc  = np.std(accs)

print(f"\nMean Accuracy : {mean_acc:.4f}")
print(f"Std Deviation : {std_acc:.4f}")
print(f"\nReport as: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
print("=" * 60)
print(f"\nAll results saved to: {RESULTS_FILE}")

Train: 40000, Val: 5000, Test: 5000


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
electra.embeddings_project.weight                 | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # 

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.155536,0.147086,0.949200,0.959487,0.937021,0.948121
2,0.110202,0.147921,0.952000,0.941920,0.962455,0.952077
3,0.059937,0.207215,0.950200,0.940316,0.960436,0.950270
4,0.038415,0.211379,0.952800,0.942361,0.963666,0.952894


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


Seed 1011 DONE: Acc=0.9552, Time=138.7min

CONCAT — 5-SEED FINAL RESULTS
Seed    42 -> Acc: 0.9538
Seed   123 -> Acc: 0.9538
Seed   456 -> Acc: 0.9544
Seed   789 -> Acc: 0.9524
Seed  1011 -> Acc: 0.9552

Mean Accuracy : 0.9539
Std Deviation : 0.0009

Report as: 95.39% ± 0.09%

All results saved to: /kaggle/working/concat_5seed_results.json
